In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset'))

In [ ]:
dataset_path = '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset'
print(os.listdir(dataset_path))

In [ ]:
image_path = os.path.join(dataset_path, 'images')

print(os.listdir(image_path)[:10])
print("Number of classes:", len(os.listdir(image_path)))

In [ ]:
first_class = os.listdir(image_path)[0]

print("Class:", first_class)
print("Images:", os.listdir(os.path.join(image_path, first_class))[:5])

In [ ]:
image_path = os.path.join(dataset_path, 'images', 'Images')

In [ ]:
classes = os.listdir(image_path)

print("Number of classes:", len(classes))
print(classes[:10])

In [ ]:
first_class = classes[0]

class_path = os.path.join(image_path, first_class)

print("Class:", first_class)
print("Number of images:", len(os.listdir(class_path)))
print(os.listdir(class_path)[:5])

In [ ]:
from keras.applications import EfficientNetB0
base_model = EfficientNetB0(
    weights = 'imagenet',
    include_top = False,
    input_shape=(224,224,3)
)

In [ ]:
base_model.trainable = False

In [ ]:
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D,Dense,Dropout 

model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(128,activation = 'relu'))
model.add(Dropout(0.3))
model.add(Dense(120,activation='softmax'))

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
from keras.utils import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    image_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='training',
    seed=42,
    shuffle=True
)

val_ds = image_dataset_from_directory(
    image_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='validation',
    seed=42,
    shuffle=False
)

In [ ]:
from keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 3,
    restore_best_weights = True
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = 10
)

In [ ]:
loss,accuracy = model.evaluate(val_ds)
print('EfficientB0 :',accuracy)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
base_model.trainable= True
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
from keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

In [ ]:
loss,accuracy = model.evaluate(val_ds)
print(' tuned EfficientB0 :',accuracy)